In [1]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000)

import shared_utils
import pandas as pd
import geopandas as gpd

import gcsfs
from calitp_data_analysis import get_fs
from calitp_data_analysis import geography_utils, utils
fs = get_fs()
import re
import google.auth
import os
import gcsfs
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

In [2]:
# GCS FILE PATH
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [3]:
with fs.open(f"{GCS_FILE_PATH}/POIS_2026.xlsx", "rb") as f:
    pois = pd.read_excel(f, sheet_name=None)

In [4]:
print(pois.keys())

dict_keys(['CENTRALCAL_POIS_2026', 'MOJAVE_POIS_2026', 'NORCAL_POIS_2026', 'SOCAL_POIS_2026'])


In [5]:
dfs = []
for sheet_name, df in pois.items():
    df = df.copy()
    df['source_sheet'] = sheet_name
    dfs.append(df)

combined = pd.concat(dfs, ignore_index = True)

In [6]:
combined.head(5)

,Region,Unnamed: 1,NAME,LAT,LON,CATEGORY,Destination_Type,CSISWeight,SSTIWeight,source_sheet
0,Central,1.0,Play Area,38.25597,-122.64955,Amusement Park,Entertainment,0.5,16.4057,CENTRALCAL_POIS_2026
1,Central,2.0,Limitless Escape Games,38.01168,-121.32213,Amusement Park,Entertainment,0.5,16.4057,CENTRALCAL_POIS_2026
2,Central,3.0,Fun Factory,38.77259,-121.26829,Amusement Park,Entertainment,0.5,16.4057,CENTRALCAL_POIS_2026
3,Central,4.0,Kelly Slater Wave Co. Surf Ranch,36.25468,-119.79029,Amusement Park,Entertainment,0.5,16.4057,CENTRALCAL_POIS_2026
4,Central,5.0,Jump Highway,38.20753,-122.13849,Amusement Park,Entertainment,0.5,16.4057,CENTRALCAL_POIS_2026


In [7]:
cols = combined.loc[:, 'NAME':'SSTIWeight'].columns

dupes = combined[combined.duplicated(subset = cols, keep = False)].sort_values(list(cols))

In [8]:
dupes.head(5)

,Region,Unnamed: 1,NAME,LAT,LON,CATEGORY,Destination_Type,CSISWeight,SSTIWeight,source_sheet
135975,Mojave,8832.0,76,34.99227,-117.54107,Petrol/Gasoline Station,Shopping,0.5,1.0,MOJAVE_POIS_2026
364617,Socal,94669.0,76,34.99227,-117.54107,Petrol/Gasoline Station,Shopping,0.5,1.0,SOCAL_POIS_2026
135951,Mojave,8808.0,76,35.06687,-118.17730,Petrol/Gasoline Station,Shopping,0.5,1.0,MOJAVE_POIS_2026
364489,Socal,94541.0,76,35.06687,-118.17730,Petrol/Gasoline Station,Shopping,0.5,1.0,SOCAL_POIS_2026
136336,Mojave,9193.0,76,35.26607,-116.07385,Petrol/Gasoline Station,Shopping,0.5,1.0,MOJAVE_POIS_2026


In [9]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460789 entries, 0 to 460788
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Region            460789 non-null  object 
 1   Unnamed: 1        351511 non-null  float64
 2   NAME              351511 non-null  object 
 3   LAT               351511 non-null  float64
 4   LON               351511 non-null  float64
 5   CATEGORY          351511 non-null  object 
 6   Destination_Type  351511 non-null  object 
 7   CSISWeight        351511 non-null  float64
 8   SSTIWeight        351511 non-null  float64
 9   source_sheet      460789 non-null  object 
dtypes: float64(5), object(5)
memory usage: 35.2+ MB


In [10]:
combined_deduped = combined.drop_duplicates(
    subset=cols
)

In [11]:
combined_deduped.info()

<class 'pandas.core.frame.DataFrame'>
Index: 341493 entries, 0 to 460788
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Region            341493 non-null  object 
 1   Unnamed: 1        341492 non-null  float64
 2   NAME              341492 non-null  object 
 3   LAT               341492 non-null  float64
 4   LON               341492 non-null  float64
 5   CATEGORY          341492 non-null  object 
 6   Destination_Type  341492 non-null  object 
 7   CSISWeight        341492 non-null  float64
 8   SSTIWeight        341492 non-null  float64
 9   source_sheet      341493 non-null  object 
dtypes: float64(5), object(5)
memory usage: 28.7+ MB


In [16]:
combined_deduped.CATEGORY.unique()

array(['Amusement Park', 'ATM', 'Bank', 'Bookstore', 'Casino', 'Cinema',
       'City Hall', 'Civic/Community Centre', 'Clothing Store',
       'Coffee Shop', 'Consumer Electronics Store',
       'Convention/Exhibition Centre', 'Court House', 'Department Store',
       'Government Office', 'Higher Education', 'Historical Monument',
       'Home Improvement & Hardware Store', 'Home Specialty Store',
       'Hospital', 'Library', 'Medical Service', 'Museum', 'Nightlife',
       'Office Supply & Services Store', 'Park/Recreation Area',
       'Performing Arts', 'Petrol/Gasoline Station', 'Pharmacy',
       'Place of Worship', 'Post Office', 'Restaurant', 'School',
       'Shopping', 'Specialty Store', 'Sporting Goods Store',
       'Tourist Attraction', nan], dtype=object)

In [12]:
combined_deduped["NAME"] = combined_deduped["NAME"].astype(str)

/tmp/ipykernel_1649/499447271.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_deduped["NAME"] = combined_deduped["NAME"].astype(str)


In [13]:
pois_gdf = gpd.GeoDataFrame(
    combined_deduped,
    geometry=gpd.points_from_xy(combined_deduped.LON, combined_deduped.LAT),
    crs="EPSG:4326"
)

In [14]:
# Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/pois_2026.parquet", "wb") as f:
    pois_gdf.to_parquet(f, index=False)